# Evaluate Clean and Adversarial Models

This notebook loads the clean and adversarial models from `models/`, evaluates test accuracy and confusion matrices on the test set, computes adversarial accuracies (FGSM and PGD), saves plots to `outputs/`, and writes a `metrics.json` file with the results.

In [3]:
# Imports and configuration
import os, json
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score

from pathlib import Path
import sys

# Ensure the project root (the directory that contains `src/`) is on sys.path so imports like `from src...` work.
repo_root = Path.cwd().resolve()
if not (repo_root / 'src').exists():
    for p in repo_root.parents:
        if (p / 'src').exists():
            repo_root = p
            break
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.config import CLEAN_MODEL_PATH, ADV_MODEL_PATH, OUTPUTS_DIR, CLASS_NAMES
from src.inference import load_model_for_inference
from src.dataset import get_dataloaders
from src.attacks import evaluate_attack
from src.utils import get_predictions, calculate_accuracy

# Ensure outputs dir exists
os.makedirs(OUTPUTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
from src.inference import load_model_for_inference
from src.dataset import get_dataloaders
from src.attacks import evaluate_attack
from src.utils import get_predictions, calculate_accuracy

# Ensure outputs dir exists
os.makedirs(OUTPUTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cpu
Using device: cpu


In [4]:
# Load dataloaders (may take a moment)
try:
    train_loader, val_loader, test_loader = get_dataloaders()
    # compute test size
    test_size = 0
    for _ in test_loader:
        test_size += 1
    print('Test batches:', test_size)
except Exception as e:
    print('Failed to create dataloaders:', e)
    raise

Found dataset at: C:\Users\shiva\Desktop\waste classifier 2.0\data\trashnet\Garbage classification\Garbage classification
Loaded 2527 images from 6 classes
Dataset splits: Train=1768, Val=379, Test=380


c:\Users\shiva\Desktop\waste classifier 2.0\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test batches: 12


In [5]:
# Load models if available
models = {}
for name, path in [('clean', CLEAN_MODEL_PATH), ('adv', ADV_MODEL_PATH)]:
    if os.path.exists(path):
        print(f'Loading {name} model from: {path}')
        models[name] = load_model_for_inference(path, device)
    else:
        print(f'Model not found: {path} (skipping)')

if not models:
    raise FileNotFoundError('No models found in models/. Please train or place the model checkpoints.')

Loading clean model from: C:\Users\shiva\Desktop\waste classifier 2.0\models\resnet_trashnet_clean.pth


c:\Users\shiva\Desktop\waste classifier 2.0\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\shiva\Desktop\waste classifier 2.0\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading adv model from: C:\Users\shiva\Desktop\waste classifier 2.0\models\resnet_trashnet_adv.pth


In [6]:
# Evaluate clean accuracy and confusion matrices for available models
results = {}
for key, model in models.items():
    print(f'Evaluating model: {key}')
    try:
        acc = calculate_accuracy(model, test_loader, device)
        print(f'Test accuracy ({key}): {acc:.2f}%')
        # get predictions to build confusion matrix
        y_true, y_pred, _ = get_predictions(model, test_loader, device)
        cm = confusion_matrix(y_true, y_pred)
        # plot and save
        fig, ax = plt.subplots(figsize=(8,6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
        ax.set_title(f'Confusion Matrix - {key}')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        fig_path = os.path.join(OUTPUTS_DIR, f'confusion_{key}.png')
        fig.tight_layout()
        fig.savefig(fig_path)
        plt.close(fig)
        print(f'Saved confusion matrix to {fig_path}')
        results[key] = {'clean_accuracy': acc, 'confusion_matrix': cm.tolist()}
    except Exception as e:
        print(f'Failed evaluating {key}:', e)
        results[key] = {'error': str(e)}

Evaluating model: clean
Test accuracy (clean): 93.16%
Saved confusion matrix to C:\Users\shiva\Desktop\waste classifier 2.0\outputs\confusion_clean.png
Evaluating model: adv


c:\Users\shiva\Desktop\waste classifier 2.0\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test accuracy (adv): 86.58%
Saved confusion matrix to C:\Users\shiva\Desktop\waste classifier 2.0\outputs\confusion_adv.png


In [7]:
# Evaluate adversarial robustness (FGSM and PGD) for each model
from src.config import EPS_FGSM, EPS_PGD, PGD_STEPS_EVAL

for key, model in models.items():
    print(f'Evaluating adversarial robustness for: {key}')
    try:
        fgsm_acc, fgsm_preds, fgsm_labels = evaluate_attack(model, test_loader, device, attack_type='fgsm', epsilon=EPS_FGSM)
        print(f'FGSM accuracy ({key}) @ eps={EPS_FGSM:.4f}: {fgsm_acc:.2f}%')
        pgd_acc, pgd_preds, pgd_labels = evaluate_attack(model, test_loader, device, attack_type='pgd', epsilon=EPS_PGD, num_steps=PGD_STEPS_EVAL)
        print(f'PGD accuracy ({key}) @ eps={EPS_PGD:.4f}, steps={PGD_STEPS_EVAL}: {pgd_acc:.2f}%')
        # store results
        results.setdefault(key, {})['fgsm_accuracy'] = float(fgsm_acc)
        results.setdefault(key, {})['pgd_accuracy'] = float(pgd_acc)
    except Exception as e:
        print(f'Failed adversarial eval for {key}:', e)
        results.setdefault(key, {})['adv_error'] = str(e)

Evaluating adversarial robustness for: clean


c:\Users\shiva\Desktop\waste classifier 2.0\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


FGSM accuracy (clean) @ eps=0.0314: 28.68%
PGD accuracy (clean) @ eps=0.0314, steps=20: 2.37%
Evaluating adversarial robustness for: adv
FGSM accuracy (adv) @ eps=0.0314: 76.58%
PGD accuracy (adv) @ eps=0.0314, steps=20: 75.53%


In [8]:
# Save metrics to outputs/metrics.json and display summary
metrics_path = os.path.join(OUTPUTS_DIR, 'metrics_from_notebook.json')
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

print('Saved metrics summary to:', metrics_path)
print('Summary:')
for k, v in results.items():
    print(k, v)

Saved metrics summary to: C:\Users\shiva\Desktop\waste classifier 2.0\outputs\metrics_from_notebook.json
Summary:
clean {'clean_accuracy': 93.15789473684211, 'confusion_matrix': [[67, 0, 0, 4, 0, 0], [0, 74, 3, 1, 1, 0], [0, 2, 63, 0, 1, 1], [0, 0, 0, 72, 4, 0], [0, 2, 1, 1, 62, 0], [0, 1, 0, 1, 3, 16]], 'fgsm_accuracy': 28.68421052631579, 'pgd_accuracy': 2.3684210526315788}
adv {'clean_accuracy': 86.57894736842105, 'confusion_matrix': [[63, 0, 1, 7, 0, 0], [0, 63, 10, 0, 4, 2], [1, 4, 57, 1, 3, 1], [0, 1, 1, 74, 0, 0], [0, 4, 2, 2, 55, 3], [1, 1, 0, 2, 0, 17]], 'fgsm_accuracy': 76.57894736842105, 'pgd_accuracy': 75.52631578947368}


## How to run

1. Open this notebook in Jupyter or VS Code and run cells in order.
2. Or run the notebook programmatically (example):

```bash
# From repository root (Windows PowerShell)
jupyter nbconvert --to notebook --execute notebooks/evaluate_models.ipynb --inplace
```

Notes:
- If model files are missing, place `resnet_trashnet_clean.pth` and/or `resnet_trashnet_adv.pth` in the `models/` directory.
- The notebook saves confusion matrix images to `outputs/` and writes `metrics_from_notebook.json` there.